# Practice 2.2 - Validation-Only Accuracy Refactor

Notebook này triển khai nhánh phát triển nhằm giảm overfit và tối ưu khả năng khái quát hóa. Toàn bộ huấn luyện, chọn cấu hình, chọn checkpoint, soft voting và test-time augmentation đều chỉ sử dụng Train và Validation. Notebook không dựng Test DataLoader và không đánh giá Final Test.

Mục tiêu trên 90% là mục tiêu thực nghiệm, không phải kết quả được bảo đảm trước khi manifest vượt qua các cổng dữ liệu và các thí nghiệm Validation được chạy đầy đủ.

## Phase 1 - Environment and Test Guard

Phase này nạp pipeline mới, xác định đường dẫn theo project root và giữ công tắc huấn luyện ở trạng thái tắt mặc định. Việc bật huấn luyện chỉ hợp lệ sau khi báo cáo cấp quyền manifest trả về `authorized=True`.

In [ ]:
import os
import sys
from pathlib import Path

working_root = Path.cwd().resolve()
project_candidates = (
    working_root,
    working_root.parent,
    working_root / "total_practice/practice_2_2",
)
project_root = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "src/practice_2_2").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Cannot locate the Practice 2.2 project root")
source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from practice_2_2.accuracy_pipeline import (
    AccuracyConfig,
    inspect_manifest_authorization,
    select_device,
    select_validation_inference_strategy,
    train_repeated_seeds,
)

manifest_path = Path(
    os.environ.get(
        "PRACTICE_2_2_MANIFEST",
        project_root / "artifacts/new_work/v3_product_visual_group_s42_v1/candidate_split_manifest.csv",
    )
)
dataset_root = Path(
    os.environ.get(
        "PRACTICE_2_2_DATASET_ROOT",
        project_root / "data/final/data_clean_balanced",
    )
)
output_root = project_root / "artifacts/new_work/accuracy_refactor_validation_only_v1"
device = select_device()
RUN_TRAINING = False

device

## Phase 2 - Manifest Authorization

Phase này chỉ đọc metadata của Train và Validation, kiểm tra quyền `use_for_model`, ảnh generated, độ phủ lớp, file tồn tại và giao nhau theo component/hash. Các hàng Test không được kiểm tra file ảnh và không được trả về cho pipeline phát triển.

In [ ]:
authorization_report = inspect_manifest_authorization(
    manifest_path,
    dataset_root,
)

authorization_report

## Phase 3 - Controlled Configuration Matrix

Ma trận giữ cố định giao thức huấn luyện hai giai đoạn và so sánh ba ứng viên: ResNet18 không label smoothing, ResNet18 có label smoothing nhẹ và EfficientNet-B0 có cùng label smoothing. Mỗi cấu hình dùng pretrained ImageNet, head warmup, sau đó chỉ mở block cuối với learning rate riêng cho backbone và head.

In [ ]:
configurations = [
    AccuracyConfig(
        architecture="resnet18",
        label_smoothing=0.0,
    ),
    AccuracyConfig(
        architecture="resnet18",
        label_smoothing=0.05,
    ),
    AccuracyConfig(
        architecture="efficientnet_b0",
        label_smoothing=0.05,
    ),
]
seeds = (42, 123, 2026)

configurations

## Phase 4 - Staged Training and Repeated Seeds

Mỗi seed bắt đầu từ pretrained weights và classifier mới có khởi tạo tái lập. Stage 1 chỉ học classifier. Checkpoint warmup tốt nhất được khôi phục trước Stage 2, nơi block cuối và classifier được fine-tune bằng AdamW, differential learning rate, cosine annealing, gradient clipping và early stopping. Checkpoint được chọn theo Validation Macro F1, sau đó Validation Accuracy và Validation Loss.

Cell chỉ chạy khi cả `RUN_TRAINING=True` và manifest đã được cấp quyền. Không có nhánh nào dựng Test DataLoader.

In [ ]:
training_summary = None
if RUN_TRAINING:
    if not authorization_report["authorized"]:
        raise RuntimeError(authorization_report["blocked_reasons"])
    training_summary = train_repeated_seeds(
        manifest_path,
        dataset_root,
        output_root,
        configurations,
        seeds=seeds,
        device=device,
    )

training_summary

## Phase 5 - Validation-Only Model Selection

Bảng tổng hợp nhiều seed xếp hạng cấu hình bằng mean Validation Macro F1, sau đó mean Validation Accuracy. Standard deviation giữa các seed cho biết độ ổn định, tránh chọn một lần chạy may mắn.

In [ ]:
configuration_ranking = (
    training_summary["summaries"]
    if training_summary is not None
    else []
)

configuration_ranking

## Phase 6 - Soft Voting and Safe TTA

Phase này cache xác suất trên Validation cho từng checkpoint và ba view bảo toàn nội dung: resize chuẩn, thu nhỏ 95% có padding và thu nhỏ 90% có padding. Không dùng horizontal flip, random crop hay phép biến đổi làm mất logo/chữ. Các tổ hợp tối đa ba model được soft vote và vẫn chỉ được chọn bằng Validation Macro F1 rồi Validation Accuracy.

Chi phí inference tăng theo số model nhân số view. Đây là phần đánh đổi tốc độ để tìm độ chính xác cao hơn mà không làm rò rỉ Final Test vào quyết định.

In [ ]:
inference_selection = None
if training_summary is not None:
    checkpoint_paths = [
        Path(result["checkpoint_path"])
        for result in training_summary["results"]
    ]
    inference_selection = select_validation_inference_strategy(
        checkpoint_paths,
        manifest_path,
        dataset_root,
        output_path=output_root / "validation_inference_selection.json",
        tta_scales=(1.0, 0.95, 0.9),
        maximum_ensemble_members=3,
        device=device,
    )

inference_selection["selected"] if inference_selection is not None else None

## Phase 7 - Decision Boundary

Kết quả của notebook này chỉ đủ để chọn cấu hình và chiến lược inference trên Validation. Final Test tiếp tục bị niêm phong. Chỉ sau khi dữ liệu được cấp quyền, protocol được đóng băng và một ứng viên duy nhất được promote thì mới có thể xin phép một bước đánh giá Final Test riêng biệt.

In [ ]:
development_decision = {
    "manifest_authorized": authorization_report["authorized"],
    "training_executed": training_summary is not None,
    "inference_strategy_selected": inference_selection is not None,
    "test_loader_constructed": False,
    "test_evaluated": False,
}

development_decision